# Black-Scholes model for option pricing

## Parameters
- Current stock price
- Amount of time until expiration (will do an array of different times)
- Strike price (same as amount of time, will do an array)
- Volatility (either array of volatilities, or later implement a function to predict it from historical data)
- Risk-free interest rate

In [1]:
# Import libraries
import pandas
import numpy
import scipy.stats
import matplotlib

## Black Scholes notes

1) Propose a series of possible prices at expiration
2) Calculate the probability of each price at expiration using the lognormal distribution
3) Calculate the payoff of the option at each price at expiration (calculate forward price at expiration, then calculate payoff)


In [31]:
# Black Scholes model for option pricing
def black_scholes(current_price, strike_price, time_to_expiration, volatility, risk_free_rate, dividend = 0):
    ## Step 1: Propose a series of possible prices at expiration
    interval = min(current_price * 0.01, 1)
    
    forward_price = current_price * numpy.exp(risk_free_rate * time_to_expiration) - dividend * time_to_expiration
    # print(f"Forward price: {forward_price:.2f}")
    possible_prices = numpy.arange(0, forward_price * 2, interval)

    ## Step 2: Calculate the probability of each price at expiration (lognormal)
    ## volatility is expected probability of price change within 1 standard deviation, yearly

    std_dev = volatility * numpy.sqrt(time_to_expiration)
    scale = numpy.exp(numpy.log(forward_price) - (std_dev**2) / 2)
    # lognormal distribution
    probabilities = scipy.stats.lognorm.pdf(possible_prices, s=std_dev, scale=scale)
    # print(f"Sum of probabilities: {numpy.sum(probabilities):.4f}")

    ## Step 3: Calculate the payoff of the option at each price at expiration

    # payoffs: call option: max(possible_price - strike_price, 0)
    call_payoffs = numpy.maximum(possible_prices - strike_price, 0)
    # expected payoff: sum of payoffs * probabilities
    call_expected_payoff = numpy.sum(call_payoffs * probabilities)
    # discount expected payoff to present value
    call_price = call_expected_payoff * numpy.exp(-risk_free_rate * time_to_expiration)

    put_payoffs = numpy.maximum(strike_price - possible_prices, 0)
    put_expected_payoff = numpy.sum(put_payoffs * probabilities)
    put_price = put_expected_payoff * numpy.exp(-risk_free_rate * time_to_expiration)

    return call_price, put_price


In [32]:
# Try the function with some parameters
current_price = 100
strike_price = [x for x in range(80, 125, 5)]
time_to_expiration = 30 / 252
volatility = 0.2
risk_free_rate = 0.01

for strike in strike_price:
    call_price, put_price = black_scholes(current_price, strike, time_to_expiration, volatility, risk_free_rate)
    print(f"Strike price: {strike}, Call price: {call_price:.2f}, Put price: {put_price:.2f}")

Strike price: 80, Call price: 20.10, Put price: 0.00
Strike price: 85, Call price: 15.12, Put price: 0.02
Strike price: 90, Call price: 10.28, Put price: 0.17
Strike price: 95, Call price: 5.97, Put price: 0.86
Strike price: 100, Call price: 2.81, Put price: 2.69
Strike price: 105, Call price: 1.02, Put price: 5.90
Strike price: 110, Call price: 0.29, Put price: 10.15
Strike price: 115, Call price: 0.06, Put price: 14.92
Strike price: 120, Call price: 0.01, Put price: 19.87


In [23]:
# Calculate number of trading days between two dates
def trading_days(start_date, end_date):
    # Create a date range
    dates = pandas.date_range(start=start_date, end=end_date, freq='B')  # 'B' frequency means business days
    return len(dates)

In [39]:
# Testing with actual market data
# Read data first
stock_price_data = pandas.read_csv("../data/raw/SPY_history.csv", parse_dates=["date"])
today_price = stock_price_data["close"].iloc[-1]
today_date = stock_price_data["date"].iloc[-1]
print(f"Today's date: {today_date.date()}, Price: {today_price:.2f}")

# Read option data (just the one file we have)
option_data = pandas.read_csv("../data/raw/SPY_options_2026-05-11.csv")

# Get expiry date from the data
expiry_date = option_data["expiry"].iloc[0]
time_to_expiration = trading_days(str(today_date.date()), expiry_date) / 252
print(f"\nExpiration date: {expiry_date}, Time to expiration: {time_to_expiration:.4f} years")

# Separate calls and puts
calls = option_data[option_data["option_type"] == "call"].copy()
puts = option_data[option_data["option_type"] == "put"].copy()

print(f"Found {len(calls)} calls, {len(puts)} puts")

# Compare for ATM options (within $15 of current price)
atm_calls = calls[abs(calls["strike"] - today_price) <= 15]
atm_puts = puts[abs(puts["strike"] - today_price) <= 15]

predicted_vol = 0.12

print("\nATM Calls:")
for _, row in atm_calls.iterrows():
    strike = row["strike"]
    market_price = row["mid"]
    model_call, _ = black_scholes(today_price, strike, time_to_expiration, predicted_vol, 0.05)
    diff = market_price - model_call
    print(f"Strike {strike}: Model={model_call:.2f}, Market={market_price:.2f}, Diff={diff:.2f}")

print("\nATM Puts:")
for _, row in atm_puts.iterrows():
    strike = row["strike"]
    market_price = row["mid"]
    _, model_put = black_scholes(today_price, strike, time_to_expiration, predicted_vol, 0.05)
    diff = market_price - model_put
    print(f"Strike {strike}: Model={model_put:.2f}, Market={market_price:.2f}, Diff={diff:.2f}")

# Test Call-Put parity for all strikes
# Call - Put = Stock Price - Strike Price * exp(-r * T)
print("\nCall-Put Parity Check:")
for strike in sorted(set(calls["strike"]).intersection(set(puts["strike"]))):
    # Check for prices from the data
    call_row = calls[calls["strike"] == strike].iloc[0]
    put_row = puts[puts["strike"] == strike].iloc[0]
    call_price = call_row["mid"]
    put_price = put_row["mid"]
    parity_diff = call_price - put_price - (today_price - strike * numpy.exp(-0.05 * time_to_expiration))
    print(f"Strike {strike}: Call={call_price:.2f}, Put={put_price:.2f}, Parity Diff={parity_diff:.2f}")
    # Check for generated prices from the model
    model_call, model_put = black_scholes(today_price, strike, time_to_expiration, predicted_vol, 0.05)
    model_parity_diff = model_call - model_put - (today_price - strike * numpy.exp(-0.05 * time_to_expiration))
    print(f"Model Call={model_call:.2f}, Model Put={model_put:.2f}, Model Parity Diff={model_parity_diff:.2f}")




Today's date: 2026-05-08, Price: 737.63

Expiration date: 2026-05-11, Time to expiration: 0.0079 years
Found 99 calls, 113 puts

ATM Calls:
Strike 723.0: Model=15.00, Market=14.61, Diff=-0.39
Strike 724.0: Model=14.03, Market=13.58, Diff=-0.45
Strike 725.0: Model=13.08, Market=12.64, Diff=-0.44
Strike 726.0: Model=12.13, Market=11.70, Diff=-0.43
Strike 727.0: Model=11.21, Market=10.80, Diff=-0.41
Strike 728.0: Model=10.30, Market=9.93, Diff=-0.37
Strike 729.0: Model=9.42, Market=8.93, Diff=-0.49
Strike 730.0: Model=8.56, Market=8.09, Diff=-0.47
Strike 731.0: Model=7.73, Market=7.21, Diff=-0.53
Strike 732.0: Model=6.94, Market=6.31, Diff=-0.63
Strike 733.0: Model=6.19, Market=5.53, Diff=-0.66
Strike 734.0: Model=5.48, Market=4.72, Diff=-0.76
Strike 735.0: Model=4.81, Market=3.96, Diff=-0.85
Strike 736.0: Model=4.19, Market=3.27, Diff=-0.93
Strike 737.0: Model=3.62, Market=2.65, Diff=-0.98
Strike 738.0: Model=3.10, Market=2.08, Diff=-1.02
Strike 739.0: Model=2.63, Market=1.60, Diff=-1.04